## Import

In [1]:
from bertopic import BERTopic
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import csv
from bertopic.vectorizers import ClassTfidfTransformer
from evaluation import evaluate_model

df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
classes = list(df["gen"])

In [2]:
# Pre-calculate embeddings
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", use_auth_token=False)
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [3]:
from bertopic.representation import KeyBERTInspired
from bertopic.representation import MaximalMarginalRelevance


# The main representation of a topic
main_representation = KeyBERTInspired()

# Additional ways of representing a topic
aspect_model2 = [KeyBERTInspired(top_n_words=30), MaximalMarginalRelevance(diversity=.5)]

# Add all models together to be run in a single `fit`
representation_model = {
   "KeyBERT": main_representation,
   "MMR":  aspect_model2 
}

In [4]:
#from sklearn.cluster import KMeans

run_name = ""

# Hyperparameters

n_neighbors = 15 # 15 -> BEST 30
n_components = 5 # 5 -> BEST 5
#random_state = [0, 1, 37, 42, 73]
random_state = [37]
min_dist = 0.0
min_cluster_size = 15 # 10 -> BEST 10
min_df = 2 # 2 -> BEST 2 BUT 1 GOOD
ngram_range = (1, 3) # (1, 2) -> BEST (1, 2)
top_n_words = 10 # 10 -> BEST 10

# Data saving

data = []

# Setup different models

cluster_model = HDBSCAN(min_cluster_size=min_cluster_size, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=min_df, ngram_range=ngram_range)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True) # enable by default --> BEST DEFAULT

# Use the representation model in BERTopic on top of the default pipeline
topic_model = BERTopic()
# Train

for seed in random_state:
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist, metric='cosine', random_state=seed)

    topic_model = BERTopic(

        # Pipeline models
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,

        # Hyperparameters
        top_n_words=top_n_words,
        n_gram_range=ngram_range,
        min_topic_size="auto", #use HDBSCAN
        verbose=True,

        # General parameters
        calculate_probabilities=True,
        language="english"
    )

    topics, probs = topic_model.fit_transform(docs, embeddings)

    # Evaluate model

    scores = evaluate_model(topic_model, docs, topics, embeddings, topk=top_n_words)  
    data.append([seed, scores[0], scores[1], scores[2], scores[3]])

df = pd.DataFrame(data, columns=["seed", "c_v", "c_npmi", "t_D", "S"])
df

2025-02-07 11:14:04,951 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-07 11:14:13,725 - BERTopic - Dimensionality - Completed ✓
2025-02-07 11:14:13,725 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-07 11:14:13,787 - BERTopic - Cluster - Completed ✓
2025-02-07 11:14:13,790 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-07 11:14:18,794 - BERTopic - Representation - Completed ✓


,seed,c_v,c_npmi,t_D,S
0,37,0.714023,0.022234,0.554545,0.516455


seed	c_v	c_npmi	t_D	S
0	37	0.801231	0.194546	0.538462	0.520185

In [92]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,-1,174,-1_gnome_gnomes_mushrooms_multiplier,"[gnome, gnomes, mushrooms, multiplier, time, r...","[number mushrooms, gnomes, gnome, colors, colo...","[number mushrooms, gnomes, combinations, pink ...",[the game seems to assign a group of gnomes to...
1,0,203,0_gnome_points_gnomes_colour,"[gnome, points, gnomes, colour, hat, brown, co...","[colour gnome, gnomes, gnome, gnome gives, col...","[colour gnome, gnome gives, yellow, scores, ro...",[Be as quick as you can when choosing the gnom...
2,1,153,1_basket_red_yellow_mushrooms,"[basket, red, yellow, mushrooms, gnomes, red b...","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, basket gnomes, mushroom...",[There are two different colours of baskets in...
3,2,105,2_mushrooms_gnome_gnomes_colour,"[mushrooms, gnome, gnomes, colour, try, gives,...","[gnome gives mushrooms, gnomes mushrooms, numb...","[gnome gives mushrooms, gnomes mushrooms, numb...",[There's a pattern when it comes to the gnomes...
4,3,97,3_points_blue_colours_pink,"[points, blue, colours, pink, purple, pattern,...","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, pattern game, s...",[It is said that certain colours score higher ...
5,4,81,4_basket_red_baskets_yellow,"[basket, red, baskets, yellow, points, gnomes,...","[gnomes baskets, colour basket, basket colours...","[gnomes baskets, basket colours, baskets red y...",[On the screen you will be presented with two ...
6,5,66,5_mushrooms_colours_mushroom_change,"[mushrooms, colours, mushroom, change, color, ...","[colors mushrooms, colour gives mushrooms, col...","[colors mushrooms, colour gives mushrooms, mus...",[There are 4 sets of mushroom people. So you ...
7,6,35,6_hats_hat_tall_short,"[hats, hat, tall, short, better, try, taller, ...","[short hats, longer hats, short yellow hat, ha...","[short hats, hat colour, hats pay, yellow hat ...","[Do your best, seems pretty random and difficu..."
8,7,24,7_keys_breaks_fingers_game,"[keys, breaks, fingers, game, make, just, sure...","[stay focused, fingers keys, keys time, focus,...","[stay focused, fingers keys, necessary keys, b...",[You really just have to go with your intuitio...
9,8,24,8_forest_mushrooms_gnomes forest_green,"[forest, mushrooms, gnomes forest, green, gnom...","[mushrooms switch forest, forest number mushro...","[mushrooms switch forest, forest number mushro...","[There are two forests. Green, orange, yellow,..."


In [93]:
topic_model.visualize_documents(docs)

## Outlier reduction

In [95]:
new_topics_prob = topic_model.reduce_outliers(docs, topics, probabilities=probs, strategy="probabilities")
new_topics_distribution = topic_model.reduce_outliers(docs, topics, strategy="distributions")
new_topics_cTFIDF = topic_model.reduce_outliers(docs, topics, strategy="c-tf-idf", threshold=0.1)
new_topics_embeddings =topic_model.reduce_outliers(docs, topics, strategy="embeddings")
new_topics = {
    "prob": new_topics_prob,
    "distribution": new_topics_distribution,
    "cTFIDF": new_topics_cTFIDF,
    "embeddings": new_topics_embeddings
    }

100%|██████████| 1/1 [00:00<00:00, 13.23it/s]


Test different topic reduction method

In [25]:
for new_topic_name in new_topics:
    topic_model.update_topics(docs, topics=new_topics[new_topic_name])
    print(evaluate_model(topic_model, docs, topics, embeddings, topk=top_n_words))

2025-02-05 18:24:27,917 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


KeyboardInterrupt: 

In [96]:
topic_model.update_topics(docs, topics=new_topics["cTFIDF"])
topic_model.get_topic_info()

2025-02-05 19:04:55,509 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


,Topic,Count,Name,Representation,KeyBERT,MMR,Representative_Docs
0,-1,11,-1_the_to_from_combinations,"[the, to, from, combinations, letter, figure o...","[number mushrooms, gnomes, gnome, colors, colo...","[number mushrooms, gnomes, combinations, pink ...",[the game seems to assign a group of gnomes to...
1,0,259,0_the_to_and_gnome,"[the, to, and, gnome, gnomes, you, of, on, it,...","[colour gnome, gnomes, gnome, gnome gives, col...","[colour gnome, gnome gives, yellow, scores, ro...",[Be as quick as you can when choosing the gnom...
2,1,164,1_basket_the_to_red,"[basket, the, to, red, yellow, you, and, mushr...","[gnomes yellow basket, gnomes red basket, bask...","[gnomes yellow basket, basket gnomes, mushroom...",[There are two different colours of baskets in...
3,2,139,2_the_mushrooms_to_of,"[the, mushrooms, to, of, you, gnome, and, gnom...","[gnome gives mushrooms, gnomes mushrooms, numb...","[gnome gives mushrooms, gnomes mushrooms, numb...",[There's a pattern when it comes to the gnomes...
4,3,112,3_the_and_to_it,"[the, and, to, it, points, that, you, blue, fo...","[certain colours, pick colour, colours, colors...","[certain colours, pick colour, pattern game, s...",[It is said that certain colours score higher ...
5,4,83,4_the_basket_to_and,"[the, basket, to, and, red, yellow, gnomes, yo...","[gnomes baskets, colour basket, basket colours...","[gnomes baskets, basket colours, baskets red y...",[On the screen you will be presented with two ...
6,5,79,5_the_mushrooms_to_you,"[the, mushrooms, to, you, and, colours, it, of...","[colors mushrooms, colour gives mushrooms, col...","[colors mushrooms, colour gives mushrooms, mus...",[There are 4 sets of mushroom people. So you ...
7,6,43,6_hats_the_and_hat,"[hats, the, and, hat, to, tall, for, short, yo...","[short hats, longer hats, short yellow hat, ha...","[short hats, hat colour, hats pay, yellow hat ...","[Do your best, seems pretty random and difficu..."
8,7,40,7_the_your_you_to,"[the, your, you, to, and, as, on, it, keys, of]","[stay focused, fingers keys, keys time, focus,...","[stay focused, fingers keys, necessary keys, b...",[You really just have to go with your intuitio...
9,8,27,8_forest_the_to_you,"[forest, the, to, you, other, the other, mushr...","[mushrooms switch forest, forest number mushro...","[mushrooms switch forest, forest number mushro...","[There are two forests. Green, orange, yellow,..."


In [ ]:
topic_model.get_topic(1, full=True)["KeyBERT"]

In [97]:
topic_model.visualize_documents(docs)

In [57]:
topic_model.visualize_hierarchy()

## DATA

Save model

In [4]:
embedding_model = "all-MiniLM-L6-v2"
topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", run_name), serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

Save data

In [5]:
param_str = str({
    "n_neighbors": n_neighbors,
    "n_components": n_components,
    "min_dist": min_dist,
    "random_state": random_state,
    "min_cluster_size": min_cluster_size,
    "min_df": min_df,
    "ngram_range": ngram_range,
    "top_n_words": top_n_words
})


data_run = [[run_name, param_str, df["Diversity"].mean(), df["Coherence"].mean()]]

df_run = pd.DataFrame(data_run, columns=["run_name", "params", "diversity", "coherence"])

df_run.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"), mode="a", header=False, index=False)

## TEST

In [18]:
from collections import Counter

# Retrieve the vectorizer model
vectorizer = topic_model.vectorizer_model

# Tokenize and count word frequencies
tokens = vectorizer.build_analyzer()(" ".join(docs))  # Tokenize all documents
word_freq = Counter(tokens)  # Count occurrences

# Convert to sorted list
sorted_word_freq = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

# Print results
print(sorted_word_freq)

[('the', 4856), ('to', 2403), ('and', 1826), ('you', 1581), ('of', 1237), ('gnomes', 987), ('that', 848), ('it', 846), ('mushrooms', 819), ('gnome', 800), ('basket', 787), ('is', 742), ('red', 721), ('yellow', 689), ('with', 599), ('will', 568), ('for', 566), ('in', 560), ('more', 545), ('colour', 544), ('on', 543), ('if', 523), ('are', 483), ('be', 470), ('points', 455), ('as', 453), ('or', 439), ('which', 422), ('blue', 414), ('one', 405), ('to the', 395), ('this', 383), ('brown', 383), ('other', 381), ('so', 381), ('colours', 379), ('purple', 369), ('green', 367), ('but', 367), ('of the', 364), ('when', 361), ('there', 333), ('pink', 330), ('then', 321), ('the other', 311), ('orange', 295), ('try', 290), ('your', 282), ('game', 277), ('out', 273), ('give', 273), ('each', 272), ('have', 263), ('on the', 262), ('time', 261), ('get', 260), ('not', 251), ('at', 249), ('the gnomes', 247), ('of mushrooms', 238), ('was', 237), ('good', 234), ('switch', 226), ('most', 224), ('can', 218), ('

## DYNAMIC

In [94]:
topics_over_time = topic_model.topics_over_time(docs, classes)

topic_model.visualize_topics_over_time(topics_over_time)

10it [00:00, 39.12it/s]


In [35]:
topic_model.get_representative_docs(2)

["There's a pattern when it comes to the gnomes' colors. One of the colors in each color pairing (green/blue, pink/brown, red/yellow, orange/purple) will render more mushrooms and it is up to you figure out which color that is. Keep selecting the color you discovered yields more mushrooms. At certain points of the game, the color that gives you the most mushrooms may change, you will know when the color you have been selecting drops in mushrooms. When that happens, select the other color. If you are unsure if you should select the other color, experiment when it is x1, better to experiment then than when it's x5. While you want to make sure you pick the correct gnome, don't hesitate. It's better to select any gnome and possibly get mushrooms than not selecting one at all. Lastly, utilize those breaks. Even if you feel like you're on a streak, your brain is a muscle and giving it time to recover builds that muscle. Good luck!",
 "It might be difficult choosing the gnome that gives the m